### Document Structure




In [166]:
from langchain_core.documents import Document

doc=Document(
    page_content="This is a test document.",
    metadata={
        "source": "test.txt",
        "author": "Ankit Mishra",
        "pages": 1
    }
)
print(doc)


page_content='This is a test document.' metadata={'source': 'test.txt', 'author': 'Ankit Mishra', 'pages': 1}


### Text Loader

In [167]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/llm_intro.txt", encoding="utf-8")
document=loader.load()
document

[Document(metadata={'source': '../data/text_files/llm_intro.txt'}, page_content='What is an LLM?\nA Large Language Model (LLM) is an artificial intelligence program trained on massive amounts of text data [1]. It uses deep learning algorithms, specifically the transformer architecture, to understand, summarize, generate, and predict new content. Instead of understanding words like humans do, an LLM converts text into numerical values called tokens. It calculates the statistical probability of which token should follow next in a sequence based on its training. This allows it to write essays, write code, and answer questions, but its knowledge is frozen at the moment its training finishes.\nWhat is RAG?\nRetrieval-Augmented Generation (RAG) is a framework that optimizes the output of an LLM by fetching information from an external knowledge base before generating a response. When a user submits a query, a RAG system searches outside sources—such as company documents, databases, or live w

### Directory Loader

In [168]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)
document=dir_loader.load()
document

[Document(metadata={'source': '..\\data\\text_files\\llm_intro.txt'}, page_content='What is an LLM?\nA Large Language Model (LLM) is an artificial intelligence program trained on massive amounts of text data [1]. It uses deep learning algorithms, specifically the transformer architecture, to understand, summarize, generate, and predict new content. Instead of understanding words like humans do, an LLM converts text into numerical values called tokens. It calculates the statistical probability of which token should follow next in a sequence based on its training. This allows it to write essays, write code, and answer questions, but its knowledge is frozen at the moment its training finishes.\nWhat is RAG?\nRetrieval-Augmented Generation (RAG) is a framework that optimizes the output of an LLM by fetching information from an external knowledge base before generating a response. When a user submits a query, a RAG system searches outside sources—such as company documents, databases, or liv

### PDF Loader

In [169]:
from langchain_community.document_loaders import PyMuPDFLoader

dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False 
)
pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-10-18T00:29:28+00:00', 'source': '..\\data\\pdf\\2307.06435v10.pdf', 'file_path': '..\\data\\pdf\\2307.06435v10.pdf', 'total_pages': 47, 'format': 'PDF 1.5', 'title': 'A Comprehensive Overview of Large Language Models', 'author': 'Humza Naveed; Asad Ullah Khan; Shi Qiu; Muhammad Saqib; Saeed Anwar; Muhammad Usman; Naveed Akhtar; Nick Barnes; Ajmal Mian;', 'subject': '', 'keywords': '', 'moddate': '2024-10-18T00:29:28+00:00', 'trapped': '', 'modDate': 'D:20241018002928Z', 'creationDate': 'D:20241018002928Z', 'page': 0}, page_content='A Comprehensive Overview of Large Language Models\nHumza Naveeda, Asad Ullah Khanb,∗, Shi Qiuc,∗, Muhammad Saqibd,e,∗, Saeed Anwarf,g, Muhammad Usmanf,g, Naveed Akhtarh,j,\nNick Barnesi, Ajmal Mianj\naThe University of Sydney, Sydney, Australia\nbUniversity of Engineering and Technology (UET), Lahore, Pakistan\ncThe Chinese University of Hong Kong (CUHK

### Chunking

In [170]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

def chunk_documents(documents):
    chunked_documents = []
    for doc in documents:
        chunks = text_splitter.split_text(doc.page_content)
        for idx, chunk in enumerate(chunks):
            metadata = dict(doc.metadata)
            metadata["chunk_index"] = idx
            chunked_documents.append(Document(page_content=chunk, metadata=metadata))
    return chunked_documents

chunks = chunk_documents(pdf_documents)
chunks
    


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-10-18T00:29:28+00:00', 'source': '..\\data\\pdf\\2307.06435v10.pdf', 'file_path': '..\\data\\pdf\\2307.06435v10.pdf', 'total_pages': 47, 'format': 'PDF 1.5', 'title': 'A Comprehensive Overview of Large Language Models', 'author': 'Humza Naveed; Asad Ullah Khan; Shi Qiu; Muhammad Saqib; Saeed Anwar; Muhammad Usman; Naveed Akhtar; Nick Barnes; Ajmal Mian;', 'subject': '', 'keywords': '', 'moddate': '2024-10-18T00:29:28+00:00', 'trapped': '', 'modDate': 'D:20241018002928Z', 'creationDate': 'D:20241018002928Z', 'page': 0, 'chunk_index': 0}, page_content='A Comprehensive Overview of Large Language Models\nHumza Naveeda, Asad Ullah Khanb,∗, Shi Qiuc,∗, Muhammad Saqibd,e,∗, Saeed Anwarf,g, Muhammad Usmanf,g, Naveed Akhtarh,j,\nNick Barnesi, Ajmal Mianj\naThe University of Sydney, Sydney, Australia\nbUniversity of Engineering and Technology (UET), Lahore, Pakistan\ncThe Chinese University 

### Embeddings

In [171]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity




In [172]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializes the embedding manager with the specified model.
        
        Args:
            model_name (str): The name of the SentenceTransformer model to use for embedding generation.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}")
            print(f"Model dimension: {self.model.get_embedding_dimension()} dimensions")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of text strings to generate embeddings for.
        
        Returns:
            np.ndarray: An array of embedding vectors for the input texts.
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

### Initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6984.18it/s]


Loaded embedding model: all-MiniLM-L6-v2
Model dimension: 384 dimensions


### Vector Store

In [173]:
class VectorStore:
    """Manages a vector store using ChromaDB for efficient similarity search."""

    def __init__(self, collection_name: str = "documents", persist_directory: str = "./chroma_db"):
        """Initializes the vector store with the specified collection name and persistence directory.
        
        Args:
            collection_name (str): The name of the ChromaDB collection to use for storing vectors.
            persist_directory (str): The directory where the ChromaDB database will be persisted.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        """Initializes the ChromaDB client and collection."""
        import os
        try:

            ### Initialize chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            ### Get or create the collection for storing document vectors
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection for storing document embeddings"}
            )
            print(f"Vector store initialized: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], metadatas: List[dict], embeddings: np.ndarray):
        if not self.collection:
            raise ValueError("Vector store collection is not initialized.")

        try:
            ids = [str(uuid.uuid4()) for _ in range(len(documents))]

            self.collection.add(
                ids=ids,
                documents=documents,          # actual text
                metadatas=metadatas,          # metadata
                embeddings=embeddings.tolist()
            )

            print(f"Added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore()
vector_store
    

Vector store initialized: documents
Existing documents in collection: 36


In [ ]:




### Store the chunks and their embeddings in the vector store
texts = [doc.page_content for doc in chunks]
metadatas = [doc.metadata for doc in chunks]
### Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings
)
vector_store.collection.count()

Generating embeddings for 356 texts...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

### Retriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    """Implements a Retrieval-Augmented Generation (RAG) retriever using ChromaDB."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initializes the RAG retriever with the specified vector store and embedding manager.
        
        Args:
            vector_store (VectorStore): An instance of the VectorStore class for managing document vectors.
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class for generating embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieves the most relevant documents for a given query.
        
        Args:
            query (str): The input query string for which to retrieve relevant documents.
            top_k (int): The number of top relevant documents to retrieve.
            score_threshold (float): The minimum cosine similarity score required for a document to be considered relevant.  
        
        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing retrieved documents with their metadata and relevance scores.
        """
        ### Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        ### Search for relevant documents in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs = []
            print(results)
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):

                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "rank": i + 1
                    })
                print(f"Retrieved {len(retrieved_docs)} relevant documents for the query: '{query}'")
            else:
                print(f"No relevant documents found for the query: '{query}'")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise


### Initialize the RAG retriever
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [ ]:
rag_retriever.retrieve("What is LLM?")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.40it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['bd985b4b-d09a-4ab8-9aa9-d1237af14649', '04c41166-e911-4945-9f93-d98c962b518c', '81712862-4d13-468b-8636-ad643e13878a', '074e7c8e-b8bb-4edb-994e-f1436918114c', '72ab1fee-9d62-4b37-a983-0bd2758506a8']], 'embeddings': None, 'documents': [['LICENSES & CERTIFICATIONS \n• \nCertified for Cybersecurity. \n• \nCertified for Large Language Models (LLM) from Google Cloud Skill Boost \n• \nCertified for Generative AI from Google Cloud Skill Boost \n• \nCertified for Applying AI Principles from Google Cloud Skill Boost \n \nSTRENGTHS \n• \nGood Problem-Solving Skills \n• \nTime Management \n• \nPassionate about Technology \n• \nCustomer Centric', 'LICENSES & CERTIFICATIONS \n• \nCertified for Cybersecurity. \n• \nCertified for Large Language Models (LLM) from Google Cloud Skill Boost \n• \nCertified for Generative AI from Google Cloud Skill Boost \n• \nCertified for Applying AI Principles from Google Cloud Skill Boost \n \nSTRENGTHS \n• \nGood P

[{'id': 'bd985b4b-d09a-4ab8-9aa9-d1237af14649',
  'content': 'LICENSES & CERTIFICATIONS \n• \nCertified for Cybersecurity. \n• \nCertified for Large Language Models (LLM) from Google Cloud Skill Boost \n• \nCertified for Generative AI from Google Cloud Skill Boost \n• \nCertified for Applying AI Principles from Google Cloud Skill Boost \n \nSTRENGTHS \n• \nGood Problem-Solving Skills \n• \nTime Management \n• \nPassionate about Technology \n• \nCustomer Centric',
  'metadata': {'trapped': '',
   'creationdate': '2026-03-07T23:57:32+05:30',
   'modDate': "D:20260307235732+05'30'",
   'title': '',
   'keywords': '',
   'creationDate': "D:20260307235732+05'30'",
   'source': '..\\data\\pdf\\Aman_Resume-2026.pdf',
   'author': 'Apache POI',
   'chunk_index': 0,
   'moddate': '2026-03-07T23:57:32+05:30',
   'total_pages': 3,
   'format': 'PDF 1.7',
   'page': 2,
   'producer': 'Microsoft® Word 2021',
   'creator': 'Microsoft® Word 2021',
   'subject': '',
   'file_path': '..\\data\\pdf\\Ama

### Integration VectorDB Context Pipeline with LLM Output

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=groq_api_key, model="openai/gpt-oss-20b")

def rag_simple(query, retreiver, llm, top_k=3):
    results = retreiver.retrieve(query, top_k=top_k)
    if not results:
        return "No relevant documents found."

    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found."
    
    ## Generate answer
    prompt=f"""Use the following retrieved documents to answer the question:
            Context: {context}

            Question: {query}

            Answer:"""
    
    response = llm.invoke(prompt)
    print(response)
    return response.content


In [ ]:
answer = rag_simple("What is LLM?", rag_retriever, llm)
answer

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.66it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['bd985b4b-d09a-4ab8-9aa9-d1237af14649', '04c41166-e911-4945-9f93-d98c962b518c', '81712862-4d13-468b-8636-ad643e13878a']], 'embeddings': None, 'documents': [['LICENSES & CERTIFICATIONS \n• \nCertified for Cybersecurity. \n• \nCertified for Large Language Models (LLM) from Google Cloud Skill Boost \n• \nCertified for Generative AI from Google Cloud Skill Boost \n• \nCertified for Applying AI Principles from Google Cloud Skill Boost \n \nSTRENGTHS \n• \nGood Problem-Solving Skills \n• \nTime Management \n• \nPassionate about Technology \n• \nCustomer Centric', 'LICENSES & CERTIFICATIONS \n• \nCertified for Cybersecurity. \n• \nCertified for Large Language Models (LLM) from Google Cloud Skill Boost \n• \nCertified for Generative AI from Google Cloud Skill Boost \n• \nCertified for Applying AI Principles from Google Cloud Skill Boost \n \nSTRENGTHS \n• \nGood Problem-Solving Skills \n• \nTime Management \n• \nPassionate about Technology \n

content='**LLM** stands for **Large Language Model**—a type of artificial‑intelligence model that is trained on vast amounts of text data to understand, generate, and manipulate natural language. These models can produce coherent text, answer questions, summarize documents, and perform a variety of language‑related tasks.' additional_kwargs={'reasoning_content': 'We have a question: "What is LLM?" We need to answer based on the retrieved documents. The documents mention "Certified for Large Language Models (LLM) from Google Cloud Skill Boost". Also they mention building an automated documentation system using LLMs. So LLM stands for Large Language Model. So answer: LLM refers to Large Language Models, a type of AI model that processes and generates natural language. Provide definition. Should be concise.'} response_metadata={'token_usage': {'completion_tokens': 164, 'prompt_tokens': 439, 'total_tokens': 603, 'completion_time': 0.178327397, 'completion_tokens_details': {'reasoning_token

'**LLM** stands for **Large Language Model**—a type of artificial‑intelligence model that is trained on vast amounts of text data to understand, generate, and manipulate natural language. These models can produce coherent text, answer questions, summarize documents, and perform a variety of language‑related tasks.'